# Teff de la secundaria desde la profundidad del tránsito + isocronas

Cadena completa:

1. `catalogo_completo_mist_sommers.csv` da, por TIC, la Teff y L de la
   componente dominante ajustadas con dos modelos (`_1` = MIST, `_2` =
   Somers17) -> de ahí sale **R1** (Stefan-Boltzmann) y la **edad del
   sistema** (`age_median_1`/`age_median_2`, se asume que ambas componentes
   son coevas).
2. `priors.csv` (de la curva de luz) da la profundidad del eclipse
   primario, `Depth_primary` = (R2/R1)^2 -> se despeja **R2**.
3. Con **R2** y la **edad** del mismo modelo, se interpola
   `Teff(R, t)` con `isocronas_interp.py` para obtener **Teff2**.

Los pasos 1-2 se repiten para MIST y para Somers17 por separado (cada
modelo tiene su propia R1, su propia edad, y se interpola contra su propia
malla de isocronas) -- así puedes comparar ambas estimaciones de Teff2, o
usar solo una de las dos.

Además (sección 5bis) se agrega **BHAC15/Baraffe**: no hay un ajuste
Teff/L propio con ese modelo en el catálogo (no genera su propio R1/R2),
pero interesa ver qué Teff le asigna esa malla a la secundaria -- se
reutiliza el (R2, edad) que ya salió de MIST y de Somers17 y se interpola
contra la malla de BHAC15.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA_MASSAGE = Path(
    "/home/luciam/Documents/starry/Resultados_Orion/datos_estrellas_massage_orion"
)
sys.path.insert(0, str(RUTA_MASSAGE))

from isocronas_interp import (
    load_isochrone_model,
    IsochroneInterpolator,
    teff_de_radio_edad_batch,
)

T_SUN = 5772.0  # K, misma constante usada en mass_age2.ipynb / exploracion_datos.ipynb

## 1) Cargar los catálogos

- `path_priors`: profundidad de tránsito por (TIC, Sector).
- `path_cat`: Teff, L y edad de la componente dominante por TIC, para MIST
  (`_1`) y Somers17 (`_2`).

In [ ]:
#path_priors = "/home/luciam/Documents/starry/Orion_yceb/priors.csv"
path_priors = "/home/luciam/Documents/starry/Resultados_Orion/priors_completosv1.csv"
path_cat = "/home/luciam/Documents/starry/Resultados_Orion/catalogo_completo_mist_sommers.csv"

priors = pd.read_csv(path_priors)
cat = pd.read_csv(path_cat)

print(f"priors: {priors.shape}, TICs únicos: {priors['TIC'].nunique()}")
print(f"cat:    {cat.shape}, TICs únicos: {cat['TIC'].nunique()}")

priors[["TIC", "Sector", "Depth_primary", "Depth_primary_err"]]

## 2) Cargar isocronas y construir los interpoladores

MIST y Somers17 (los dos que tiene `catalogo_completo_mist_sommers.csv`, con
su propio R1/edad) y además BHAC15/Baraffe: no hay un ajuste Teff/L propio
con BHAC15 en el catálogo, pero igual nos interesa esa malla para
interpolar la Teff de la **secundaria** -- se reutiliza el R2 y la edad que
ya salieron de MIST o de Somers17 (ver sección 5bis).

In [ ]:
modelos_dir = RUTA_MASSAGE / "modelos"

grid_mist = load_isochrone_model(
    "MIST", modelos_dir / "iso_mist" / "MIST_iso_6a9a687177c08_todas_edades.csv"
)
grid_somers = load_isochrone_model(
    "Somers17", modelos_dir / "iso_somers17" / "f017track_ms.csv"
)
grid_bhac15 = load_isochrone_model(
    "BHAC15", modelos_dir / "iso_barafe" / "BHAC15_iso.txt"
)

interp_mist = IsochroneInterpolator(grid_mist, "MIST")
interp_somers = IsochroneInterpolator(grid_somers, "Somers17")
interp_bhac15 = IsochroneInterpolator(grid_bhac15, "BHAC15")

## 3) R1 (Stefan-Boltzmann) y configuración por modelo

`R1/Rsun = sqrt(L/Lsun) * (T_sun/Teff)^2`, igual que en `mass_age2.ipynb` /
`exploracion_datos.ipynb`. Se usan las columnas `_median` (más robustas
frente a colas asimétricas que `_mean`), que es lo que ya usaba
`exploracion_datos.ipynb` al construir `R2_s`/`R2_mist`.

`MODELOS` es el punto donde se elige qué modelo(s) usar: deja los dos para
comparar, o comenta uno para quedarte solo con MIST o solo con Somers17.

In [ ]:
def radio_stefan_boltzmann(Teff, logL, T_sun=T_SUN):
    """R/Rsun a partir de Teff [K] y log10(L/Lsun)."""
    return 10 ** (logL / 2) * (T_sun / Teff) ** 2


MODELOS = {
    "MIST": dict(T_col="T_median_1", L_col="lgL_median_1", age_col="age_median_1", interp=interp_mist),
    "Somers17": dict(T_col="T_median_2", L_col="lgL_median_2", age_col="age_median_2", interp=interp_somers),
}

for nombre, cfg in MODELOS.items():
    cat[f"R1_{nombre}"] = radio_stefan_boltzmann(cat[cfg["T_col"]], cat[cfg["L_col"]])

cat[["TIC", "R1_MIST", "age_median_1", "R1_Somers17", "age_median_2"]]

## 4) R2 desde la profundidad del tránsito

`Depth_primary = (R2/R1)^2` -> `R2 = R1 * sqrt(Depth_primary)`.

Se junta `priors` (una fila por TIC+Sector) con `cat` (una fila por TIC) por
`TIC`: R1 y la edad se repiten para todos los sectores del mismo TIC, la
profundidad sí puede variar de un sector a otro.

In [ ]:
cols_cat = ["TIC"] + [f"R1_{n}" for n in MODELOS] + [cfg["age_col"] for cfg in MODELOS.values()]
datos = priors.merge(cat[cols_cat], on="TIC", how="left")

print(f"Filas priors: {len(priors)}  ->  filas tras el merge: {len(datos)}")
print(f"TICs de priors sin match en el catálogo: "
      f"{sorted(set(priors['TIC']) - set(cat['TIC']))}")

for nombre in MODELOS:
    datos[f"R2_{nombre}"] = datos[f"R1_{nombre}"] * np.sqrt(datos["Depth_primary"])

datos[["TIC", "Sector", "Depth_primary", "R1_MIST", "R2_MIST", "R1_Somers17", "R2_Somers17"]]

## 5) Teff2 = Teff(R2, t) por isocrona

Para cada modelo, se interpola con su propia malla, en su propia edad
(`age_median_1` para MIST, `age_median_2` para Somers17). Fuera del rango
de la malla (p. ej. edades > 100 Myr en la malla de MIST cargada aquí, o
masas fuera de la cobertura de Somers17) el resultado es NaN, no una
extrapolación -- ver `isocronas_interp.py`.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)  # fuera de rango -> NaN, se resume abajo
    for nombre, cfg in MODELOS.items():
        datos[f"Teff2_{nombre}"] = teff_de_radio_edad_batch(
            datos[f"R2_{nombre}"], datos[cfg["age_col"]], cfg["interp"]
        )

for nombre in MODELOS:
    n_ok = datos[f"Teff2_{nombre}"].notna().sum()
    print(f"  Teff2_{nombre}: {n_ok} / {len(datos)} filas dentro de la malla")

datos[[
    "TIC", "Sector", "Depth_primary",
    "R2_MIST", "age_median_1", "Teff2_MIST",
    "R2_Somers17", "age_median_2", "Teff2_Somers17",
]]

## 5bis) Teff2 con la malla de Baraffe (BHAC15)

BHAC15 no tiene un ajuste Teff/L propio en `catalogo_completo_mist_sommers.csv`
(no hay columnas `_3`), así que no genera su propio R1/R2 -- pero sí nos
interesa saber qué Teff le asigna la malla de Baraffe al **mismo** par
(R2, edad) que ya salió de MIST y de Somers17. Por eso se evalúa dos veces,
una reutilizando `(R2_MIST, age_median_1)` y otra `(R2_Somers17,
age_median_2)`, cada una contra la malla de BHAC15.

In [ ]:
FUENTES_R2 = {
    "MIST": ("R2_MIST", "age_median_1"),
    "Somers17": ("R2_Somers17", "age_median_2"),
}

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    for nombre, (r2_col, age_col) in FUENTES_R2.items():
        datos[f"Teff2_BHAC15_desde_{nombre}"] = teff_de_radio_edad_batch(
            datos[r2_col], datos[age_col], interp_bhac15
        )

cols_bhac15 = [f"Teff2_BHAC15_desde_{n}" for n in FUENTES_R2]
for c in cols_bhac15:
    print(f"  {c}: {datos[c].notna().sum()} / {len(datos)} filas dentro de la malla")

datos[[
    "TIC", "Sector",
    "R2_MIST", "age_median_1", "Teff2_MIST", "Teff2_BHAC15_desde_MIST",
    "R2_Somers17", "age_median_2", "Teff2_Somers17", "Teff2_BHAC15_desde_Somers17",
]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(datos))
etiquetas = [f"TIC {t}\nS{s}" for t, s in zip(datos["TIC"], datos["Sector"])]

ax.scatter(x - 0.15, datos["Teff2_MIST"], label="Teff2 (MIST)", color="tab:blue")
ax.scatter(x - 0.05, datos["Teff2_BHAC15_desde_MIST"], label="Teff2 (BHAC15, con R2/t de MIST)",
           color="tab:blue", marker="x")
ax.scatter(x + 0.05, datos["Teff2_Somers17"], label="Teff2 (Somers17)", color="tab:orange")
ax.scatter(x + 0.15, datos["Teff2_BHAC15_desde_Somers17"], label="Teff2 (BHAC15, con R2/t de Somers17)",
           color="tab:orange", marker="x")

ax.set_xticks(x)
ax.set_xticklabels(etiquetas, fontsize=8)
ax.set_ylabel("Teff2 = Teff(R2, t) [K]")
ax.set_title("Teff de la secundaria a partir de la profundidad del eclipse")
ax.legend(fontsize=8)
ax.grid(alpha=0.25, linestyle="--")
fig.tight_layout()
plt.show()

## 6) Guardar

In [ ]:
salida_path = RUTA_MASSAGE / "teff2_desde_profundidad_transito.csv"
datos.to_csv(salida_path, index=False)
print(f"Guardado: {salida_path}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Asegurar orden: agrupado por TIC y luego por Sector
datos = datos.sort_values(["TIC", "Sector"]).reset_index(drop=True)

# 2. Espaciado por grupo: dejamos un "hueco" extra cada vez que cambia el TIC
tic_vals = datos["TIC"].values
group_gap = 1.0   # separación normal dentro del grupo
extra_gap = 0.8   # espacio extra entre grupos de TIC distintos

x = np.zeros(len(datos))
for i in range(1, len(datos)):
    x[i] = x[i-1] + group_gap + (extra_gap if tic_vals[i] != tic_vals[i-1] else 0)

fig, ax = plt.subplots(figsize=(max(8, len(datos) * 0.9), 5))

offset = 0.15
ax.scatter(x - offset, datos["Teff2_MIST"], label="Teff2 (MIST)", color="tab:blue")
ax.scatter(x - offset/2, datos["Teff2_BHAC15_desde_MIST"], label="Teff2 (BHAC15, con R2/t de MIST)",
           color="tab:blue", marker="x")
ax.scatter(x + offset/2, datos["Teff2_Somers17"], label="Teff2 (Somers17)", color="tab:orange")
ax.scatter(x + offset, datos["Teff2_BHAC15_desde_Somers17"], label="Teff2 (BHAC15, con R2/t de Somers17)",
           color="tab:orange", marker="x")

# 3. Etiquetas de dos niveles: TIC arriba (una vez por grupo), Sector abajo (por punto)
ax.set_xticks(x)
ax.set_xticklabels([f"S{s}" for s in datos["Sector"]], fontsize=8)

# TIC como etiqueta secundaria, centrada en cada grupo, y líneas divisorias
change_idx = np.where(np.r_[True, tic_vals[1:] != tic_vals[:-1]])[0]
group_bounds = list(change_idx) + [len(datos)]

for start, end in zip(group_bounds[:-1], group_bounds[1:]):
    centro = x[start:end].mean()
    ax.text(centro, -0.12, f"TIC {tic_vals[start]}", transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=9, fontweight="bold")
    if start > 0:
        # línea divisoria entre grupos
        limite = (x[start-1] + x[start]) / 2
        ax.axvline(limite, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)

ax.set_ylabel("Teff2 = Teff(R2, t) [K]")
ax.set_title("Teff de la secundaria a partir de la profundidad del eclipse")
ax.legend(fontsize=8)
ax.grid(alpha=0.25, linestyle="--")
ax.margins(x=0.02)
fig.subplots_adjust(bottom=0.2)  # espacio extra para la etiqueta de TIC
plt.show()